In [76]:
# 환경설정
import os
import sys
import time
from tqdm import tqdm
import nest_asyncio
nest_asyncio.apply()
from dotenv import load_dotenv

load_dotenv()
# duckdb
import duckdb

# 데이터 전처리
import pandas as pd
import numpy as np
import polars as pl
from datetime import datetime, timedelta
from copy import deepcopy

# 데이터 수집
import requests
from bs4 import BeautifulSoup

# VectorDB 저장
from hashlib import md5
from langchain_community.vectorstores.utils import filter_complex_metadata # ChromaDB가 제공하지 못하는 데이터 형태를 자동으로 string처리


## LLM 활용
from summary_function import NewsSummaryAgent
# LLM 활용을 위한 dict형태 구축
from collections import defaultdict

# langchain 계열
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

from langchain_openai import ChatOpenAI

# 1. LLM 모델 세팅 (OpenAI)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
#from summary_function_openai import NewsSummaryAgent, build_summary_graph  # 너가 만든 것
from summart_function_openai_2 import NewsSummaryAgent, build_summary_graph  # 너가 만든 것
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.3)

In [2]:
ETF_conn = duckdb.connect('../DB/ETF.db')

In [3]:
ETF_df = ETF_conn.execute('select * from IRP_ETF_COMPOSE_table').fetchdf()
ETF_conn.close()

In [4]:
ETF_df = ETF_df.map(lambda x : x.strip())

In [5]:
ETF_df_task_1 = ETF_df[ETF_df['구성종목 종목명'] != "설정현금액"]
ETF_df_task_1 = ETF_df_task_1[ETF_df_task_1['구성종목 종목명'] != "원화현금"]
ETF_df_task_1 = ETF_df_task_1[1:]

In [6]:
# 원하는 컬럼 필터링
ETF_df_task_2 = ETF_df_task_1[['ETF 종목명','구성종목 표준코드','구성종목 종목명','편입비율']]

In [7]:
# 구성종목 중 상위 5개 추출
ETF_df_task_3 = ETF_df_task_2.sort_values(by=['ETF 종목명','편입비율'], ascending=False)

In [8]:
# 종목별 상위 5개
ETF_df_task_4= ETF_df_task_3.groupby('ETF 종목명').head(5)

In [9]:
ETF_df_task_4

,ETF 종목명,구성종목 표준코드,구성종목 종목명,편입비율
7839,파워 코스피100,KR7005930003,삼성전자,22.979477
7899,파워 코스피100,KR7105560007,KB금융,2.846638
7856,파워 코스피100,KR7012450003,한화에어로스페이스,2.680652
7876,파워 코스피100,KR7035420009,NAVER,2.595262
7836,파워 코스피100,KR7005380001,현대차,2.280681
...,...,...,...,...
59727,1Q 25-08 회사채(A+이상)액티브,KR6079317D93,JB 우리캐피탈488-3(지),8.851804
59728,1Q 25-08 회사채(A+이상)액티브,KR6095923D95,현대커머셜485-3(지),8.850334
59723,1Q 25-08 회사채(A+이상)액티브,KR6023788D91,신한캐피탈487-2,8.844456
59722,1Q 25-08 회사채(A+이상)액티브,KR601945CD90,아이비케이캐피탈290-7,8.842961


In [10]:
ETF_df_task_4[ETF_df_task_4['ETF 종목명'].str.contains('반도체')]

,ETF 종목명,구성종목 표준코드,구성종목 종목명,편입비율
61034,WON 반도체밸류체인액티브,KR7402340004,SK스퀘어,4.673431
61026,WON 반도체밸류체인액티브,KR7039030002,이오테크닉스,4.617986
61027,WON 반도체밸류체인액티브,KR7058470006,리노공업,4.522346
61019,WON 반도체밸류체인액티브,KR7000150003,두산,4.518261
61030,WON 반도체밸류체인액티브,KR7140860008,파크시스템스,4.513108
...,...,...,...,...
59981,ACE AI반도체포커스,KR7000660001,SK하이닉스,25.755918
59985,ACE AI반도체포커스,KR7005930003,삼성전자,24.736614
59992,ACE AI반도체포커스,KR7042700005,한미반도체,24.473018
59998,ACE AI반도체포커스,KR7140860008,파크시스템스,1.519344


In [11]:
ETF_df_task_4[ETF_df_task_4['ETF 종목명'].str.contains('전지')]

,ETF 종목명,구성종목 표준코드,구성종목 종목명,편입비율
44379,TIGER 글로벌리튬&2차전지SOLACTIVE(합성),KRYZTRSEAH18,글로벌리튬2차전지 TRS 241017-18,8.990678
44380,TIGER 글로벌리튬&2차전지SOLACTIVE(합성),KRYZTRSF1E02,글로벌리튬2차전지 TRS 250114-02,32.05569
44381,TIGER 글로벌리튬&2차전지SOLACTIVE(합성),KRYZTRSF5E01,글로벌리튬2차전지 TRS 250514-01,27.325525
44378,TIGER 글로벌리튬&2차전지SOLACTIVE(합성),KRYZTRSE7Q01,글로벌리튬2차전지 TRS 240724-01,2.877921
44377,TIGER 글로벌리튬&2차전지SOLACTIVE(합성),KRYZTRSE7C04,글로벌리튬2차전지 TRS 240712-04,14.624746
...,...,...,...,...
43926,ACE 2차전지&친환경차액티브,KR7012330007,현대모비스,8.46777
43920,ACE 2차전지&친환경차액티브,KR7005380001,현대차,8.23665
43914,ACE 2차전지&친환경차액티브,KR7000270009,기아,8.090785
43921,ACE 2차전지&친환경차액티브,KR7005490008,POSCO홀딩스,7.842668


In [12]:
ETF_df_task_4[ETF_df_task_4['ETF 종목명'].str.contains('자율주행')]

,ETF 종목명,구성종목 표준코드,구성종목 종목명,편입비율
44360,TIGER 글로벌자율주행&전기차SOLACTIVE,US5949181045,MICROSOFT CORP,3.65966
44363,TIGER 글로벌자율주행&전기차SOLACTIVE,US67066G1040,NVIDIA CORP,3.405895
44366,TIGER 글로벌자율주행&전기차SOLACTIVE,US7475251036,QUALCOMM INC,2.930486
44342,TIGER 글로벌자율주행&전기차SOLACTIVE,US02079K3059,ALPHABET INC-CL A,2.869684
44318,TIGER 글로벌자율주행&전기차SOLACTIVE,JP3633400001,TOYOTA MOTOR CORP,2.808817
43543,KODEX 자율주행액티브,KR7012330007,현대모비스,8.590932
43570,KODEX 자율주행액티브,KR7307950006,현대오토에버,7.626326
43529,KODEX 자율주행액티브,KR7000660001,SK하이닉스,5.967903
43556,KODEX 자율주행액티브,KR7086280005,현대글로비스,4.925796
43531,KODEX 자율주행액티브,KR7005380001,현대차,4.391058


In [13]:
ETF_df_task_4[ETF_df_task_4['ETF 종목명'].str.contains('금융')]

,ETF 종목명,구성종목 표준코드,구성종목 종목명,편입비율
34483,TIGER 25-12 금융채(AA-이상),KR6205498EC7,하나카드273,4.988738
34398,TIGER 25-12 금융채(AA-이상),KR6005273F23,아이엠뱅크46-02이12A-21,4.10297
34473,TIGER 25-12 금융채(AA-이상),KR6140178EB5,케이비국민카드421-1,3.300191
34461,TIGER 25-12 금융채(AA-이상),KR6079314EA3,JB 우리캐피탈524-1(지),2.485386
34475,TIGER 25-12 금융채(AA-이상),KR6145763DC9,BNK캐피탈338-3,2.48222
7606,TIGER 200 금융,KR7316140003,우리금융지주,7.825771
7587,TIGER 200 금융,KR7000810002,삼성화재,7.048532
7595,TIGER 200 금융,KR7032830002,삼성생명,5.727802
7607,TIGER 200 금융,KR7323410001,카카오뱅크,5.180046
7602,TIGER 200 금융,KR7138040001,메리츠금융지주,4.741424


## 고객 데이터 시나리오

In [14]:
Customer_A = ETF_df_task_4[ETF_df_task_4['ETF 종목명'].isin(['ACE AI반도체포커스','ACE 2차전지&친환경차액티브','KODEX 자율주행액티브','RISE 200금융'])]

In [15]:
Customer_A = Customer_A.reset_index().drop('index',axis = 1)

In [16]:
Customer_A

,ETF 종목명,구성종목 표준코드,구성종목 종목명,편입비율
0,RISE 200금융,KR7316140003,우리금융지주,7.848761
1,RISE 200금융,KR7000810002,삼성화재,7.198696
2,RISE 200금융,KR7032830002,삼성생명,5.759096
3,RISE 200금융,KR7323410001,카카오뱅크,5.192032
4,RISE 200금융,KR7138040001,메리츠금융지주,4.756095
5,KODEX 자율주행액티브,KR7012330007,현대모비스,8.590932
6,KODEX 자율주행액티브,KR7307950006,현대오토에버,7.626326
7,KODEX 자율주행액티브,KR7000660001,SK하이닉스,5.967903
8,KODEX 자율주행액티브,KR7086280005,현대글로비스,4.925796
9,KODEX 자율주행액티브,KR7005380001,현대차,4.391058


In [17]:
cus_news = duckdb.connect('../DB/Customer_news.db')

In [18]:
cusA_news_df = cus_news.execute('select * from articles').fetch_df()
cus_news.close()

In [19]:
cusA_news_df.head(5)

,header,summary,content,url,datetime,ticker
0,"18일, 거래소 외국인 순매수상위에 전기,전자 업종 4종목",None,"외국인 투자자는 18일 거래소에서 삼성전자, NAVER, 크래프톤 등을 중점적으로 ...",https://www.hankyung.com/article/202506184343L,2025.06.18 18:35,우리금융지주
1,"""실적·주주환원 훈풍""…은행·증권주 신고가 행진, 스탁론 매수세도 유입",None,국내 은행 및 증권주들이 2분기 실적 호조와 주주환원 기대감에 힘입어 강세를 이어가...,https://www.hankyung.com/article/202507096279a,2025.07.09 10:30,우리금융지주
2,"12일, 외국인 거래소에서 한화에어로스페이스(+5.3%), 현대차(+0.25%) 등...",None,"외국인 투자자는 12일 거래소에서 한화에어로스페이스, 현대차, 현대건설 등을 중점적...",https://www.hankyung.com/article/202506121791L,2025.06.12 18:35,우리금융지주
3,"금감원, 우리금융 경평 '2→3등급' 결론…이번주 통보할 듯",None,금감원 / 사진=노정동 기자\n\n 금융감독원이 우리금융...,https://www.hankyung.com/article/2025031784556,2025.03.17 11:18,우리금융지주
4,한도 초과 걱정 없이 연 4%대 금리로 저점 집중투자 시작하기!,None,부자네스탁론이 특별 이벤트로 5년고정 연 4.9%의 저금리 스탁론 상품을 출시하면서...,https://www.hankyung.com/article/202506056123a,2025.06.05 14:38,우리금융지주


In [20]:
cusA_news_df.shape

(10000, 6)

## 날짜 전처리

In [21]:
cusA_news_df['Date'] = cusA_news_df.datetime.str[:10]
cusA_news_df = cusA_news_df.drop('datetime',axis=1)

In [22]:
cusA_news_df['Date'] = pd.to_datetime(cusA_news_df['Date'])
# 2. 기준일 계산 (오늘 날짜 - 5일)
today = pd.to_datetime("2025-07-30")
five_days_ago = today - timedelta(days=30)

# 3. 최근 5일치 필터링
recent_news_df = cusA_news_df[cusA_news_df['Date'] >= five_days_ago]

In [23]:
recent_news_df= recent_news_df.drop_duplicates()

In [24]:
# 최근 30일치 뉴스 기사 필터링 시 남아있는 뉴스 기사 개수
recent_news_df.groupby('ticker').count()

,header,summary,content,url,Date
ticker,,,,,
DB하이텍,30,4,30,30,30
LG에너지솔루션,500,96,500,500,500
POSCO홀딩스,464,23,464,464,464
SK하이닉스,500,119,500,500,500
기아,411,191,411,411,411
메리츠금융지주,51,5,51,51,51
삼성생명,140,45,140,140,140
삼성전자,500,161,500,500,500
삼성화재,96,31,96,96,96


## VectorDB 저장

- 진행한 종목 : 우리금융지주, 삼성전자

In [25]:
# 4. OpenAI 임베딩 모델 로딩
embedding = OpenAIEmbeddings(model="text-embedding-3-large")

persist_directory = "../VectorDB/chroma_news_db"

vectordb = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding
)

In [32]:
def pick_splitter_by_length(text_len: int) -> RecursiveCharacterTextSplitter:
    """
    뉴스 본문의 길이에 따라 적절한 텍스트 분할기를 반환합니다.
    """
    if text_len <= 1200:
        # 짧은 기사 → 굳이 자르지 않고 1덩어리로 처리
        return RecursiveCharacterTextSplitter(
            chunk_size=1200,
            chunk_overlap=0,
            separators=["\n\n", "\n", " ", ""]
        )
    elif text_len <= 10_000:
        # 중간 길이 → 일반적인 1,200자 기준으로 분할
        return RecursiveCharacterTextSplitter(
            chunk_size=1200,
            chunk_overlap=150,
            separators=["\n\n", "\n", " ", ""]
        )
    elif text_len <= 50_000:
        # 긴 기사 → 덩어리를 좀 더 키움
        return RecursiveCharacterTextSplitter(
            chunk_size=1800,
            chunk_overlap=200,
            separators=["\n\n", "\n", " ", ""]
        )
    else:
        # 초장문 → 더 크게 자르되, 요약도 고려 (이건 후속 처리 필요)
        return RecursiveCharacterTextSplitter(
            chunk_size=2000,
            chunk_overlap=200,
            separators=["\n\n", "\n", " ", ""]
        )
    

In [33]:
# 2. 문서 리스트 생성 (chunk + metadata 포함)
def make_documents(df):
    docs = []

    for idx, row in df.iterrows():
        text = row["content"]
        splitter = pick_splitter_by_length(len(text))
        chunks = splitter.split_text(text)

        for i, chunk in enumerate(chunks):
            metadata = {
                "title": row["header"],
                "url": row["url"],
                "Date": row["Date"],
                "ticker": row.get("ticker", "None"),
                "chunk_idx": i,
                "original_idx": idx,
            }
            docs.append(Document(page_content=chunk, metadata=metadata))

    return docs


In [34]:
def get_recent_articles(df: pd.DataFrame, ticker: str, days: int = 5):
    df['Date'] = pd.to_datetime(df['Date'])
    today = df['Date'].max()
    recent_df = df[
        (df['ticker'] == ticker) &
        (df['Date'] >= today - timedelta(days=days))
    ]
    return recent_df.sort_values(by="Date", ascending=False)


In [35]:
def make_doc_id(d: Document) -> str:
    """
    url + chunk_idx(없으면 0) 조합으로 안정적인 고유 id 생성
    """
    base = f"{d.metadata.get('url','')}_{d.metadata.get('chunk_idx', 0)}"
    return md5(base.encode("utf-8")).hexdigest()

def chunks(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]


In [36]:
recent_df_woori = get_recent_articles(cusA_news_df,ticker='삼성전자',days = 30)
recent_df_woori['Date'] = recent_df_woori.Date.astype('str')
recent_df_woori_docs = make_documents(recent_df_woori)
BATCH = 64  # 상황에 맞게 조절

for docs in tqdm(chunks(recent_df_woori_docs, BATCH), total=(len(recent_df_woori_docs) + BATCH - 1) // BATCH):

    ids = [make_doc_id(d) for d in docs]
    vectordb.add_documents(documents=docs, ids=ids)

100%|██████████| 15/15 [00:37<00:00,  2.47s/it]


In [43]:
len(recent_df_woori_docs)

219

In [37]:
print("Number of documents in DB:", vectordb._collection.count())

Number of documents in DB: 1139


## cross-encoder 모델(w/langchain 예시)

In [27]:
retriever = vectordb.as_retriever(search_kwargs={"k": 30,'filter' : {'ticker' : '삼성전자'}})

In [28]:
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [f"Document {i + 1}:\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

In [29]:
query = '''
이 뉴스들 중에서 "삼성전자"가 핵심 주제로 다뤄진 기사만 알려줘.
다른 회사 언급이 많거나, 삼성전자가 단순히 함께 언급된 기사라면 제외해줘.
'''
docs = retriever.invoke(query)
pretty_print_docs(docs)

Document 1:

플랙트(공조), 마시모(오디오), 젤스(헬스케어) 등 여러 인수합병을 진행했습니다.앞으로도 M&A를 통해 급변하는 글로벌 기술 트렌드에 대응하겠다면서 AI, 공조, 메디텍, 전장 등 신성장 분야의 후보 업체들을 검토 중이라고 설명했습니다. 사업 경쟁력 강화와 사업 규모를 확대하기 위해 비슷한 분야의 업체들을 추가로 인수할 가능성을 내비친 것으로 보입니다.또한 삼성은 올해 상반기 미래 신기술, 우수 기술업체 발굴을 위해 약 40여개 업체에 1억2,000만 달러(1,700억원) 이상을 벤처 투자했다고 강조했습니다.지난 29일에는 국내 AI 반도체 기업인 리벨리온이 삼성 계열사로부터 대규모 투자를 받았다고 발표하기도 했습니다.이재용 회장이 사법리스크 해소로 '뉴삼성' 구축에 속도를 내고 있는 가운데, 지난 2017년 하만과 같은 초대형 M&A도 기대되고 있습니다.<앵커>잘 들었습니다.홍헌표기자 hphong@wowtv.co.kr
----------------------------------------------------------------------------------------------------
Document 2:

자사주 소각, 이재용 회장의 사법 리스크 해소 등 여러 호재에도 불구하고 삼성전자 주가는 줄곧 6만 원대에 머물렀다.삼성전자가 사실상 ‘잠든 상태’에서도 코스피는 크게 올랐다. 그렇다면 삼성전자가 본격적으로 기지개를 켠다면 어떨까.지난 28일 삼성전자는 테슬라와의 납품 계약, 이른바 ‘23조 잭팟’ 소식에 주가가 7만 원대를 회복했다. 전문가들은 “삼성전자의 반등이야말로 코스피 5000 돌파의 열쇠”라고 진단한다.삼성전자가 지수를 실질적으로 견인하려면 주가는 얼마나 올라야 할까. 삼성전자의 시가총액은 7월 24일 기준 약 390조원이다.2021년 고점(주가 약 9만1000원) 당시에는 약 540조원에 달했다. 단순 계산으로 지수 3300에서 5000까지 51% 오르려면, 삼성전자 역시 현재 주가에서 9만원 이상으로 회복

In [30]:
docs[0].metadata

{'Date': '2025-07-31',
 'ticker': '삼성전자',
 'original_idx': 5953,
 'url': 'https://www.hankyung.com/article/2025073141065',
 'title': '삼성 "HBM4 샘플 출하…품목관세는 예의주시"',
 'chunk_idx': 3}

In [31]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

#model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
compressor = CrossEncoderReranker(model=model, top_n=50)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
)

/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [32]:
CrossEncoder_prompt = '''
이 뉴스들 중에서 "삼성전자"가 핵심 주제로 다뤄진 기사만 알려줘.
다른 회사 언급이 많거나, 삼성전자가 단순히 함께 언급된 기사라면 제외해줘.
'''

In [33]:
compressed_docs = compression_retriever.invoke(CrossEncoder_prompt)
pretty_print_docs(compressed_docs)

/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Document 1:

자사주 소각, 이재용 회장의 사법 리스크 해소 등 여러 호재에도 불구하고 삼성전자 주가는 줄곧 6만 원대에 머물렀다.삼성전자가 사실상 ‘잠든 상태’에서도 코스피는 크게 올랐다. 그렇다면 삼성전자가 본격적으로 기지개를 켠다면 어떨까.지난 28일 삼성전자는 테슬라와의 납품 계약, 이른바 ‘23조 잭팟’ 소식에 주가가 7만 원대를 회복했다. 전문가들은 “삼성전자의 반등이야말로 코스피 5000 돌파의 열쇠”라고 진단한다.삼성전자가 지수를 실질적으로 견인하려면 주가는 얼마나 올라야 할까. 삼성전자의 시가총액은 7월 24일 기준 약 390조원이다.2021년 고점(주가 약 9만1000원) 당시에는 약 540조원에 달했다. 단순 계산으로 지수 3300에서 5000까지 51% 오르려면, 삼성전자 역시 현재 주가에서 9만원 이상으로 회복해야 한다는 계산이 나온다.삼성전자와 함께 주가를 견인할 신흥 대장주도 필요하다. 기준은 명확하다. 시가총액이 크고 기관·외국인 수급이 몰리며 정부 정책이나 글로벌 산업 트렌드를 타야 한다. 2025년 7월까지의 상승률 상위 종목 중에는 한화에어로스페이스, LIG넥스원, 두산에너빌리티, SK하이닉스, POSCO홀딩스 등이 후보로 꼽힌다. 이들은 정책, 실적, 수급, 산업 구조 재편이라는 4박자를 동시에 갖춘 기업들이다.정채희 기자 poof34@hankyung.com
----------------------------------------------------------------------------------------------------
Document 2:

삼성전자가 올해 하반기 확장현실(XR) 헤드셋(사진), 두 번 접는 ‘트리폴드폰’ 등 신제품을 출격시킨다. 인공지능(AI), 메디테크(의료기술), 냉난방공조(HVAC) 등 신사업을 강화하기 위해 추가 인수합병(M&A)도 추진한다.31일 삼성전자에 따르면 갤럭시 스마트폰 등 모바일 기기를 담당하는 모바일경험(MX)사업부는 올 2분기 매출 29조2000억원, 영업이

In [34]:
compressed_docs[0]

Document(id='d2b36cf9dccce2895bc8ae765f857eec', metadata={'Date': '2025-07-29', 'ticker': '삼성전자', 'url': 'https://magazine.hankyung.com/business/article/202507249472b', 'title': '지수 5000의 열쇠, 삼성전자? [코스피 5000을 외치는 사람들]', 'original_idx': 5709, 'chunk_idx': 3}, page_content='자사주 소각, 이재용 회장의 사법 리스크 해소 등 여러 호재에도 불구하고 삼성전자 주가는 줄곧 6만 원대에 머물렀다.삼성전자가 사실상 ‘잠든 상태’에서도 코스피는 크게 올랐다. 그렇다면 삼성전자가 본격적으로 기지개를 켠다면 어떨까.지난 28일 삼성전자는 테슬라와의 납품 계약, 이른바 ‘23조 잭팟’ 소식에 주가가 7만 원대를 회복했다. 전문가들은 “삼성전자의 반등이야말로 코스피 5000 돌파의 열쇠”라고 진단한다.삼성전자가 지수를 실질적으로 견인하려면 주가는 얼마나 올라야 할까. 삼성전자의 시가총액은 7월 24일 기준 약 390조원이다.2021년 고점(주가 약 9만1000원) 당시에는 약 540조원에 달했다. 단순 계산으로 지수 3300에서 5000까지 51% 오르려면, 삼성전자 역시 현재 주가에서 9만원 이상으로 회복해야 한다는 계산이 나온다.삼성전자와 함께 주가를 견인할 신흥 대장주도 필요하다. 기준은 명확하다. 시가총액이 크고 기관·외국인 수급이 몰리며 정부 정책이나 글로벌 산업 트렌드를 타야 한다. 2025년 7월까지의 상승률 상위 종목 중에는 한화에어로스페이스, LIG넥스원, 두산에너빌리티, SK하이닉스, POSCO홀딩스 등이 후보로 꼽힌다. 이들은 정책, 실적, 수급, 산업 구조 재편이라는 4박자를 동시에 갖춘 기업들이다.정채희 기자 poof34@hankyung.com')

### original 본문 가져오기

In [35]:
def get_full_article_from_chroma(original_idx: int, kind : str, vectordb) -> dict:
    """original_idx 기준으로 chunk들을 모아 원문 복원"""
    # 1. 해당 article의 모든 chunk 가져오기
    VecDB = vectordb._collection
    results = VecDB.get(
      where = { "$and" : [ # 빈 쿼리로 전체 탐색
            {"original_idx": original_idx}, 
            {"ticker": kind}
            ]
         },
         include = ['documents','metadatas']   
        )
    print(f'results : {results}')
    if not results:
        return {"error": f"No chunks found for original_idx {original_idx}"}

    #print(results)
    #print(results['metadatas'][0]['chunk_idx'])

    # 4. 대표 metadata 하나 뽑아 저장
    return {
        "title": results['metadatas'][0]["title"],
        "url": results['metadatas'][0]["url"],
        "Date": results['metadatas'][0]["Date"],
        "ticker": results['metadatas'][0]["ticker"],
        "content": results['documents'][0]
    }

In [36]:
original_idxs = list(map(lambda x: x.metadata['original_idx'], compressed_docs))

In [37]:
original_idxs

[5709,
 5942,
 5914,
 5809,
 5680,
 5557,
 5922,
 5798,
 5615,
 5674,
 5565,
 5574,
 5907,
 5643,
 5533,
 5682,
 5815,
 5896,
 5660,
 5867,
 5953,
 5693,
 5797,
 5882,
 5713,
 5953,
 5515,
 5584,
 5685,
 5673]

In [38]:
top_n_original = [get_full_article_from_chroma(idx,kind='삼성전자',vectordb=vectordb) for idx in original_idxs]

results : {'ids': ['0915dd135d88715d463dd3a01c7a4e2f', 'b81c8d01d976bd79a521a0ca3984cf22', 'c478c40b605cae0db62fc1bee8e01d04', 'd2b36cf9dccce2895bc8ae765f857eec'], 'embeddings': None, 'documents': ['BBIG에서 지금조방원으로, 코스피 주도주의 변화\r\n                                \n\n\n\n28일 서울 여의도 한국거래소에 삼성전자 종가가 70,400원을 나타내고 있다. 이날 코스피 지수는 전장보다 13.47포인트(0.42%) 오른 3,209.52에 거래를 마쳤다. 25.7.28 /한국경제신문 이솔 기자‘신고가’의 계절이다. 지난 6월 한 달에만 상장주 5개 종목 중 1개꼴로 52주 신고가를 갈아치웠다. 코스피는 연초 대비 32.72% 상승하며 3300선 돌파를 목표로 하고 있다. 혹자는 “코스피200에서 눈감고 골라도 수익이 나는 장세”라고 말할 정도다.그런데 이상하다. 코스피지수가 역사적 고점을 향해가고 있지만, 웃지 못하는 이들이 많다. ‘국민주’ 삼성전자가 지난 28일 간만에 기지개를 켜며 7만 원대를 회복했지만, 2021년 고점에 물린 개미 투자자 상당수는 여전히 손실 구간에 머물러 있다.NH투자증권에 따르면 해당 플랫폼 기준 삼성전자 보유자 중 손실은 본 투자자는 무려 62.11%에 달한다. 한때 ‘차세대 국민주’로 주목받았던 카카오(손실 투자자 비율 89.09%)와 네이버(81.24%), LG화학(84.59%) 등도 상황은 비슷하다. “20년 만에 진짜 랠리가 왔다는데 내 계좌는 왜 이 모양일까.”유동성 장세와 정책 장세', '결론부터 말하자면 주도주가 바뀌었다.2021년 시장을 지배했던 ‘BBIG’(배터리·바이오·인터넷·게임)는 힘을 잃었다. 2025년의 시장은 ‘지금조방원’(지주회사·금융·조선·방산·원자력)과 ‘ABCDE’(AI·바이오·문화·방산·에너지)의 

In [39]:
top_n_original[0]

{'title': '지수 5000의 열쇠, 삼성전자? [코스피 5000을 외치는 사람들]',
 'url': 'https://magazine.hankyung.com/business/article/202507249472b',
 'Date': '2025-07-29',
 'ticker': '삼성전자',
 'content': 'BBIG에서 지금조방원으로, 코스피 주도주의 변화\r\n                                \n\n\n\n28일 서울 여의도 한국거래소에 삼성전자 종가가 70,400원을 나타내고 있다. 이날 코스피 지수는 전장보다 13.47포인트(0.42%) 오른 3,209.52에 거래를 마쳤다. 25.7.28 /한국경제신문 이솔 기자‘신고가’의 계절이다. 지난 6월 한 달에만 상장주 5개 종목 중 1개꼴로 52주 신고가를 갈아치웠다. 코스피는 연초 대비 32.72% 상승하며 3300선 돌파를 목표로 하고 있다. 혹자는 “코스피200에서 눈감고 골라도 수익이 나는 장세”라고 말할 정도다.그런데 이상하다. 코스피지수가 역사적 고점을 향해가고 있지만, 웃지 못하는 이들이 많다. ‘국민주’ 삼성전자가 지난 28일 간만에 기지개를 켜며 7만 원대를 회복했지만, 2021년 고점에 물린 개미 투자자 상당수는 여전히 손실 구간에 머물러 있다.NH투자증권에 따르면 해당 플랫폼 기준 삼성전자 보유자 중 손실은 본 투자자는 무려 62.11%에 달한다. 한때 ‘차세대 국민주’로 주목받았던 카카오(손실 투자자 비율 89.09%)와 네이버(81.24%), LG화학(84.59%) 등도 상황은 비슷하다. “20년 만에 진짜 랠리가 왔다는데 내 계좌는 왜 이 모양일까.”유동성 장세와 정책 장세'}

In [40]:
original_contents = list(map(lambda x: x['content'],top_n_original))

In [41]:
total_contents = ''.join(original_contents)

In [42]:
total_contents

'BBIG에서 지금조방원으로, 코스피 주도주의 변화\r\n                                \n\n\n\n28일 서울 여의도 한국거래소에 삼성전자 종가가 70,400원을 나타내고 있다. 이날 코스피 지수는 전장보다 13.47포인트(0.42%) 오른 3,209.52에 거래를 마쳤다. 25.7.28 /한국경제신문 이솔 기자‘신고가’의 계절이다. 지난 6월 한 달에만 상장주 5개 종목 중 1개꼴로 52주 신고가를 갈아치웠다. 코스피는 연초 대비 32.72% 상승하며 3300선 돌파를 목표로 하고 있다. 혹자는 “코스피200에서 눈감고 골라도 수익이 나는 장세”라고 말할 정도다.그런데 이상하다. 코스피지수가 역사적 고점을 향해가고 있지만, 웃지 못하는 이들이 많다. ‘국민주’ 삼성전자가 지난 28일 간만에 기지개를 켜며 7만 원대를 회복했지만, 2021년 고점에 물린 개미 투자자 상당수는 여전히 손실 구간에 머물러 있다.NH투자증권에 따르면 해당 플랫폼 기준 삼성전자 보유자 중 손실은 본 투자자는 무려 62.11%에 달한다. 한때 ‘차세대 국민주’로 주목받았던 카카오(손실 투자자 비율 89.09%)와 네이버(81.24%), LG화학(84.59%) 등도 상황은 비슷하다. “20년 만에 진짜 랠리가 왔다는데 내 계좌는 왜 이 모양일까.”유동성 장세와 정책 장세삼성전자가 올해 하반기 확장현실(XR) 헤드셋(사진), 두 번 접는 ‘트리폴드폰’ 등 신제품을 출격시킨다. 인공지능(AI), 메디테크(의료기술), 냉난방공조(HVAC) 등 신사업을 강화하기 위해 추가 인수합병(M&A)도 추진한다.31일 삼성전자에 따르면 갤럭시 스마트폰 등 모바일 기기를 담당하는 모바일경험(MX)사업부는 올 2분기 매출 29조2000억원, 영업이익 3조1000억원을 기록했다. 1년 전보다 매출은 6.6%, 영업이익은 39% 늘었다. 갤럭시 S25 시리즈 등 프리미엄 제품과 갤럭시탭 10 시리즈 등 태블릿PC 판매가 늘어난 영향이다.삼성전자는 7월 출시한 갤럭시 Z폴드7·플립7 

## 요약하기

### 요약함수 호출

버전 2

In [54]:
from summart_function_openai_2 import NewsSummaryAgent, build_summary_graph  # 너가 만든 것

In [55]:
def summarize_top_articles_2(total_contents: str,ticker:str) -> pd.DataFrame:
    # doc : 10개 > max_iter = 3
    # doc : 30 개 > max_iter = 5
    # doc : 20개 >   max_iter = 5
    agent = NewsSummaryAgent(max_iters=10)
    runnable = build_summary_graph(agent)

    rows = []
    doc = total_contents

    state = {"article": doc, "summary": "", "feedback": "", "iteration": 0}
    result = runnable.invoke(state)

    print("✅ 실행 결과 키:", result.keys())
    # 여기서 final_summary 반드시 존재해야 함(위 패치 기준)
    final_summary = result.get("final_summary")
    if not final_summary:
        print("❌ final_summary 없음. 디버그용 전체 상태:", result)
        # 계속 진행할지, 실패로 표기할지 선택

    rows.append({
        "ticker": ticker,
        "date": '2025-07-30', # 위 조회 기준일자로 연동시켜서 바꿀 예정
        "summary": final_summary,
        "feedback": result.get("last_feedback", "피드백 없음"),
    })

    return pd.DataFrame(rows)


In [56]:
result_df = summarize_top_articles_2(total_contents,ticker='삼성전자')

summart : ✅ 주요 요약
- 삼성전자, 테슬라와 22조 원 규모 반도체 위탁생산 계약 체결
  삼성전자는 테슬라와 22조 원 규모의 반도체 위탁생산 계약을 체결했으며, 이는 삼성의 새로운 텍사스 공장에서 테슬라의 차세대 AI6 칩을 생산하는 계약이다.

- 삼성전자, 다양한 신제품 출시 및 신사업 강화 계획
  삼성전자는 XR 헤드셋, 트리폴드폰 등 혁신 제품을 연말까지 출시할 예정이며, AI, 메디테크, HVAC 등 신사업 강화를 위해 추가 M&A를 추진 중이다.

- 삼성전자, 2분기 실적 발표 및 반도체 사업 성장 전략
  삼성전자는 2분기 매출 74조 6천억 원, 영업이익 4조 7천억 원을 기록했으며, 반도체 사업에서는 HBM3E와 DDR5 제품 판매 비중을 확대하고 있다.

🔑 키워드
- 삼성전자
- 테슬라
- 반도체 위탁생산
- AI6 칩
- XR 헤드셋
- 트리폴드폰
- M&A
- HVAC
- 2분기 실적
- HBM3E
- DDR5
[should_stop] Iteration: 0
[should_stop] Feedback:
 - 정확성: 좋음
- 포괄성: 부족함 <reason> [요약에서 삼성전자의 주가 변동이나 투자자 반응과 관련된 내용이 빠져 있습니다. 또한, 코스피 지수와 관련된 정보도 포함되지 않았습니다.] </reason>
- 간결성: 좋음
- 문장구성: 좋음
- 일관성: 좋음

피드백:
포괄성 측면에서, 요약에 삼성전자의 주가 변동이나 투자자 반응에 대한 정보가 포함되면 더 좋을 것입니다. 예를 들어, 삼성전자의 주가가 테슬라와의 계약 소식에 어떻게 반응했는지, 그리고 코스피 지수의 변동 상황도 요약에 추가되면 독자가 더 포괄적인 이해를 할 수 있을 것입니다. 이러한 추가 정보는 요약의 깊이를 더하고 독자에게 더 많은 배경 정보를 제공합니다.
[should_stop] next_step = no
====== result : refined_summary='✅ 주요 요약\n\n- **삼성전자, 테슬라와 22조 원 규모 반도체 위탁생산 계

In [46]:
result_df

,ticker,date,summary,feedback
0,삼성전자,2025-07-30,"✅ 주요 요약\n\n- **삼성전자, 테슬라와 22조 원 규모 반도체 위탁생산 계약...","- 정확성: 좋음\n <reason> [요약 내용이 원문 기사와 일치하며, 주요 ..."


In [ ]:
# doc =50, max_iter = 10, 일관성 넣으니 3번 돌고 끝!
print(result_df.summary.values[0])

✅ 주요 요약

- **삼성전자, 테슬라와 22조 원 규모 반도체 위탁생산 계약 체결**  
  삼성전자는 테슬라와의 대규모 계약을 통해 AI6 칩을 생산하며, 이는 삼성의 파운드리 사업 확대에 기여할 것으로 기대됨.

- **삼성전자 주가 상승, 11개월 만에 7만 원대 회복**  
  테슬라와의 계약 소식으로 삼성전자 주가는 급등하며 7만 원대를 회복, 이는 투자자들의 긍정적인 반응을 이끌어냄.

- **삼성전자, 신제품 출시 및 M&A로 미래 성장 동력 강화**  
  삼성전자는 XR 헤드셋과 트리폴드폰 등 차세대 혁신 제품을 연말까지 출시할 계획이며, AI, 메디테크, HVAC 등 신사업을 강화하고 다양한 분야에서 M&A를 추진하여 성장 동력을 확보하고 있음.

- **삼성전자 2분기 실적 발표, 반도체 사업의 도전과 성과**  
  삼성전자는 2분기 매출 74조6000억원, 영업이익 4조7000억원을 기록했으며, 반도체 사업은 4000억원의 영업이익을 거두며 적자를 면함. 이는 메모리 사업의 재고 자산 평가 충당금과 비메모리 사업의 대중 제재 영향으로 인한 도전 과제가 있었음.

🔑 키워드
- 삼성전자
- 테슬라
- 반도체 위탁생산
- AI6 칩
- 파운드리 사업
- 주가 상승
- M&A
- 신제품 출시
- 신사업 강화
- 2분기 실적
- 반도체 사업 성과 및 도전


In [ ]:
# doc : 20개, max_iter = 5, 일관성 평가항목 존재
print(result_df.summary.values[0])

✅ 주요 요약

- **삼성전자, 테슬라와 22조 원 규모 반도체 위탁생산 계약 체결**  
  삼성전자는 테슬라와의 대규모 반도체 위탁생산 계약을 통해 AI6 칩을 생산하며, 이는 삼성의 파운드리 사업 확장에 중요한 역할을 할 것으로 예상됨.

- **삼성전자 주가, 테슬라 계약 소식에 7만 원대 회복**  
  테슬라와의 계약 발표 후 삼성전자 주가는 급등하며 11개월 만에 7만 원대를 회복, 투자자들의 긍정적 반응을 이끌어냄.

- **삼성전자, 2분기 실적 발표: 영업이익 4조7000억 원 기록**  
  삼성전자는 2분기 매출 74조6000억 원, 영업이익 4조7000억 원을 기록하며 전년 동기 대비 영업이익이 55.23% 감소했으나, 반도체 사업에서 적자를 면함.

- **삼성전자, 신성장 분야 M&A 및 혁신 제품 출시 계획**  
  삼성전자는 AI, 메디테크, HVAC 등 신사업 강화를 위해 추가 M&A를 추진하며, XR 헤드셋과 트리폴드폰 등 혁신 제품 출시로 시장 경쟁력을 높일 계획임.

🔑 키워드
- 삼성전자
- 테슬라
- 반도체 위탁생산
- AI6 칩
- 파운드리
- 주가 상승
- 2분기 실적
- M&A
- 혁신 제품
- XR 헤드셋
- 트리폴드폰


In [ ]:
# doc : 30 , max_iter = 5, 평가항목 일관성 넣기 전
print(result_df.summary.values[0])

삼성전자는 최근 테슬라와 22조 원 규모의 반도체 위탁생산 계약을 체결하며 주가가 급등, 11개월 만에 '7만전자'를 회복했습니다. 이번 계약은 삼성의 텍사스 공장에서 테슬라의 차세대 AI6 칩을 생산하는 것으로, 이는 삼성의 파운드리 사업 확장에 중요한 전환점이 될 것으로 보입니다. 이 계약은 삼성의 기술력과 대외 신뢰도를 높이며, 글로벌 반도체 시장에서의 입지를 강화할 것으로 기대됩니다.

삼성전자는 AI, 메디테크, 냉난방공조(HVAC) 등 신사업 강화를 위해 추가 인수합병(M&A)을 추진 중이며, XR 헤드셋과 트리폴드폰 등 혁신 제품 출시를 계획하고 있습니다. 이러한 신제품들은 삼성의 기술력과 시장 경쟁력을 높이는 데 기여할 것으로 기대됩니다. 특히, AI 기능을 적용한 제품들은 삼성의 미래 성장 동력으로 자리 잡을 전망입니다.

2분기 실적 발표에서는 매출 74조6000억원, 영업이익 4조7000억원을 기록했으나, 전년 대비 영업이익이 55.23% 감소하며 시장 기대치를 밑돌았습니다. 이는 반도체 사업의 재고 자산 평가 충당금과 비메모리 사업의 대중 제재 영향 때문으로 분석됩니다.

코스피 지수는 삼성전자의 강세에 힘입어 상승세를 보였으나, 여전히 많은 투자자들이 손실을 보고 있는 상황입니다. 삼성전자 외에도 카카오, 네이버, LG화학 등 주요 기업들이 손실을 기록하고 있으며, 이는 시장의 불확실성을 반영합니다.

삼성전자의 이번 계약과 신사업 확장 계획은 향후 성장 가능성을 높이며, 주가 상승의 주요 요인으로 작용하고 있습니다. 시장은 삼성의 파운드리 사업 확장과 AI 관련 제품의 성공 가능성에 주목하고 있으며, 이는 삼성의 장기적인 성장 전략에 긍정적인 영향을 미칠 것으로 보입니다. 또한, 삼성전자는 HVAC 사업 확대를 위해 독일의 플랙트그룹을 인수하는 등 적극적인 M&A 활동을 통해 신성장 동력을 확보하고 있습니다.


In [ ]:
# 10개 추출했을 때의 요약문, 일관성 넣기 전
print(result_df.summary.values[0])

✅ 주요 요약

- **삼성전자, 대규모 반도체 위탁생산 계약 체결**
  삼성전자는 약 22.8조원 규모의 반도체 위탁생산 계약을 체결하며, 테슬라와의 AI 반도체 공급 계약을 통해 반도체 사업 확장을 기대하고 있습니다. 이는 삼성 반도체 역사상 최대 규모의 계약으로 주목받고 있습니다.

- **다양한 신제품 출시 및 M&A로 사업 다각화 추진**
  삼성전자는 XR 헤드셋, 트리폴드폰 등 혁신 제품을 출시하고, AI, 메디테크, HVAC 등 신사업 강화를 위해 적극적인 M&A를 추진 중입니다. 특히, HVAC 사업 확장을 위해 독일의 플랙트그룹을 인수하며, 관련 시장 공략을 강화하고 있습니다.

- **반도체 소부장주, 삼성전자 수주 소식에 급등**
  삼성전자의 테슬라 AI 칩 수주 소식으로 반도체 소부장 관련주들이 급등했으며, 이는 기관투자가들의 매수세를 이끌어내고 있습니다.

🔑 키워드
- 삼성전자
- 반도체 위탁생산
- 테슬라
- AI 칩
- M&A
- XR 헤드셋
- HVAC 사업
- 반도체 소부장주
- 기관투자가

**개선 포인트:**
- 삼성전자의 HVAC 사업 확장 및 관련 M&A 활동을 포함하여 포괄성을 강화했습니다. 이를 통해 독자가 삼성전자의 전반적인 전략을 더 잘 이해할 수 있도록 했습니다.


# 과거 버전

In [204]:
import pandas as pd
from datetime import datetime, timedelta
# cross-encoder
from sentence_transformers import CrossEncoder
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from summary_function_openai import NewsSummaryAgent, build_summary_graph  # 너가 만든 것

# 1. 모델 준비 (CrossEncoder for Re-ranking) # 예시 모델 하나 생성
rerank_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# 2. 최근 5일치 필터링 함수
def get_recent_articles(df: pd.DataFrame, ticker: str, days: int = 5):
    df['Date'] = pd.to_datetime(df['Date'])
    today = df['Date'].max()
    recent_df = df[
        (df['ticker'] == ticker) &
        (df['Date'] >= today - timedelta(days=days))
    ]
    return recent_df.sort_values(by="Date", ascending=False)

# 3. huggingface 모델 이용.
def rerank_articles(df: pd.DataFrame,ticker:str ,query: str, top_k: int = 5):
    task_df= df[df.ticker == ticker]
    docs = task_df['content'].tolist()
    # 모델 이용?? 
    pairs = [(query, doc) for doc in docs]
    scores = rerank_model.predict(pairs)
    
    task_df_2 = task_df.copy()
    task_df_2['score'] = scores
    return task_df_2.sort_values(by='score', ascending=False).head(top_k)

# 4. 전체 요약 실행 함수
def summarize_top_articles(df: pd.DataFrame, ticker: str, query: str, top_k: int = 5):
    recent_df = get_recent_articles(df, ticker)
    top_df = rerank_articles(recent_df, query=query,ticker=ticker, top_k=top_k)

    agent = NewsSummaryAgent()
    runnable = build_summary_graph(agent)

    results = []
    for _, row in top_df.iterrows():
        state = {
            "article": row['content'],
            "summary": "",
            "feedback": "",
            "iteration": 0
        }
        result = runnable.invoke(state)


        print("✅ 실행 결과 타입:", type(result))
        print("✅ 실행 결과 키 목록:", result.keys())
        print("✅ 실행 결과 전체 내용:", result)

        if "final_summary" not in result:
            print("❌ final_summary 키가 없습니다. 중단합니다.")
            continue  # 또는 raise Exception("final_summary 없음")

        print(f'실행 결과 : {result}')
        results.append({
            "ticker": row['ticker'],
            "date": row['Date'],
            "header": row['header'],
            "url": row['url'],
            "summary": result["final_summary"], 
             "feedback": result.get("last_feedback", "피드백 없음")
        })
    return pd.DataFrame(results)


In [55]:
query = "우리금융지주 관련 시황"
ticker = "우리금융지주"  # 예시
result_df = summarize_top_articles(cusA_news_df, ticker=ticker, query=query, top_k=3)

/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


[should_stop] Iteration: 1
[should_stop] Feedback:
 - 정확성: 좋음 <reason> 원문의 내용을 정확하게 반영하고 있음. </reason>
- 포괄성: 부족함 <reason> 원문에서 언급된 '기술 경쟁력 확보, 고객사 확대, 수익구조 개선 등 실질적 사업성과로 뒷받침돼야 한다는 지적' 등의 중요한 내용이 누락되었음. </reason>
- 간결성: 좋음 <reason> 불필요한 표현 없이 요약 내용을 간결하게 전달하였음. </reason>
- 문장구성: 좋음 <reason> 문장이 자연스럽고 명확하게 구성되어 있음. </reason>

[피드백]
요약의 포괄성이 부족한 점이 아쉽습니다. 원문에서 언급된 '기술 경쟁력 확보, 고객사 확대, 수익구조 개선 등 실질적 사업성과로 뒷받침돼야 한다는 지적' 등의 중요한 내용을 요약에 포함시키면 더욱 완벽한 요약이 될 것 같습니다. 이 부분을 고려하여 요약을 수정해보시는 것을 추천드립니다.
✅ '정확성' 평가 통과
❌ '포괄성' 평가에서 좋음이 아님
[should_stop] next_step = no


KeyError: 'Input to PromptTemplate is missing variables {\'"foo"\', \'"properties"\'}.  Expected: [\'"foo"\', \'"properties"\', \'article\', \'feedback\', \'summary\'] Received: [\'article\', \'summary\', \'feedback\']\nNote: if you intended {"foo"} to be part of the string and not a variable, please escape it with double curly braces like: \'{{"foo"}}\'.\nFor troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/INVALID_PROMPT_INPUT '

In [35]:
def summarize_top_articles(total_contents :str):

    agent = NewsSummaryAgent()
    runnable = build_summary_graph(agent)
    results = []
    
    doc = total_contents

    state = {
        "article": doc['content'],
        "summary": "",
        "feedback": "",
        "iteration": 0
    }
    result = runnable.invoke(state)


    print("✅ 실행 결과 타입:", type(result))
    print("✅ 실행 결과 키 목록:", result.keys())
    print("✅ 실행 결과 전체 내용:", result)

    if "final_summary" not in result:
        print("❌ final_summary 키가 없습니다.")

    print(f'실행 결과 : {result}')
    results.append({
        "ticker": doc['ticker'],
        "date": doc['Date'],
        "header": doc['title'],
        "url": doc['url'],
        "summary": result["final_summary"], 
            "feedback": result.get("last_feedback", "피드백 없음")
    })
    return pd.DataFrame(results)
